# 📝 Neo4j 시작하기 과제 LV2(응용): 결과 가공(pandas·collections)

> 제공된 조회 결과를 **pandas·collections 로 가공**해 분포와 순위를 뽑습니다. 집계는 Cypher 가 아니라 **파이썬(pandas·Counter)으로** 계산합니다(그게 이 과제의 훈련).

## 풀이 방법
1. 준비 셀 → 점검 셀 → 도구 import 셀을 먼저 실행하세요.
2. 각 문제의 **제공 셀**이 데이터를 가져옵니다. 여러분은 그 결과를 pandas·Counter 로 가공합니다.
3. **자가채점 셀**로 확인하세요(✅ 통과!). 6번은 서술형입니다.

- 답은 **제공 결과를 파이썬으로 가공해** 만드세요. 출력을 눈으로 읽어 옮겨 적으면 채점은 지나가도 이 과제의 훈련이 되지 않습니다.

화이팅!

아래 준비 셀·점검 셀·도구 셀을 먼저 실행하세요.

In [ ]:
# Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 이 단원은 그래프를 조회만 합니다(그래프를 바꾸지 않습니다).
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase   # 파이썬용 공식 드라이버. 이 클래스로 접속 통로를 연다

# 1) 접속 정보 읽기: .env 에 적힌 값을 환경변수로 올린다(파일이 없으면 조용히 넘어간다)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 두 번째 인자는 .env 에 그 키가 없을 때 쓰는 기본값이다(로컬 Desktop 의 표준 주소·사용자).
# .env 를 못 읽어도 에러가 아니라 이 값으로 조용히 넘어가니, 이 셀 마지막 줄에 찍히는
# 주소가 실습 전용 DB 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
# 2) 드라이버 만들기: 접속 통로 하나를 노트북 전체가 나눠 쓴다(쿼리마다 새로 만들지 않는다)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 실제로 붙어 본다. 인스턴스가 꺼져 있거나 비밀번호가 틀리면 여기서 에러가 난다


# 3) 수업 내내 쓰는 헬퍼: 쿼리를 보내고 결과를 파이썬 자료형으로 바꿔 준다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    with driver.session() as session:
        # 세션은 with 블록을 벗어나면 자동으로 닫힌다. record.data() 가 결과 한 행을 dict 로 바꾼다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)   # 이 줄이 찍히면 연결까지 성공한 것이다

In [ ]:
# Movies 그래프가 적재돼 있는지 점검: 실행만 하세요(그래프를 바꾸지 않습니다).
# MATCH (n) 은 레이블을 가리지 않고 모든 노드를 고른다. count(n) 결과는 한 행이라 [0] 으로 dict 를 꺼낸다
_n = run_cypher("MATCH (n) RETURN count(n) AS cnt")[0]["cnt"]   # 이름 앞 밑줄은 이 셀에서만 쓰는 임시 변수라는 표시
print("연결된 그래프의 노드 수:", _n)   # 171 이면 준비 완료, 0 이면 아직 적재 전이다

In [ ]:
# 이 과제에서 쓸 도구(실행만 하세요).
import pandas as pd
from collections import Counter

## 데이터 살펴보기
아래 셀은 **실행만** 하세요. 이번 과제에서 가공할 영화 데이터를 먼저 훑어봅니다.

In [ ]:
# 영화 제목·개봉연도를 DataFrame 으로 훑어봅니다
movies_df = pd.DataFrame(run_cypher("MATCH (m:Movie) RETURN m.title AS title, m.released AS released"))
display(movies_df.head())
print('영화 편수:', len(movies_df))

## 1. 개봉연도를 10년 단위로 묶어 분포 보기
**배경**: 영화가 어느 시대에 몰려 있는지 보려면 개봉연도를 **10년 단위(decade)** 로 묶어 셉니다.

**요구사항**:
- 제공된 `movies_df` 에 `released` 를 10년 단위로 내림한 **`decade`** 열을 추가하세요(예: 1999 → 1990, 2003 → 2000). 계산은 `(연도 // 10) * 10`.
- 각 decade 의 영화 수를 세어, **1990년대(1990)** 의 편수를 변수 **`n_1990s`** 에 담으세요.

**예시**: `movies_df` 에 `decade` 열이 추가되고, `n_1990s` 는 정수 하나입니다(1990년대가 가장 많은 시대라는 것만 알려 드립니다. 편수는 직접 세세요).

<details><summary>힌트</summary>

```text
접근방법:
- 정수 나눗셈으로 decade 열을 만들고, value_counts 로 센 뒤 1990 값을 꺼낸다.

세부구현:
1. movies_df 에 decade 열을 추가한다. released 를 10 으로 정수 나눗셈(//)한 뒤 다시 10 을 곱한다.
2. decade 열에 value_counts 를 적용해 decade 별 편수를 얻는다.
3. 그 결과에서 1990 에 해당하는 값을 정수로 꺼내 n_1990s 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] decade 열을 실제로 만들었는지, 값이 맞는지 함께 봅니다.
assert 'decade' in movies_df.columns, 'movies_df 에 decade 열을 추가하세요'
assert (movies_df['decade'] % 10 == 0).all(), 'decade 값이 10년 단위(10 의 배수)가 아닙니다'
assert (movies_df['decade'] == (movies_df['released'] // 10) * 10).all(), \
    'decade 열 값이 개봉연도를 10년 단위로 내린 값과 다릅니다'
assert n_1990s == int((movies_df['decade'] == 1990).sum()), \
    '1990년대 편수가 맞지 않습니다. decade 열에서 1990 인 행을 세었는지 확인하세요'
print('✅ 통과!')

## 2. 가장 많은 영화에 출연한 배우
**배경**: 출연 관계(`ACTED_IN`)를 배우별로 세면 다작 배우를 찾을 수 있습니다. 제공 셀이 **(배우, 영화) 쌍 전체**를 가져옵니다.

**요구사항**:
- `Counter` 로 배우별 출연 편수를 세어, **가장 많이 출연한 배우 한 명**의 `(이름, 편수)` 를 변수 **`top_actor`** 에 담으세요(**`(이름, 편수)` 튜플**).

**예시**: `top_actor` 는 `('이름', 편수)` 형태의 튜플입니다(1위는 2위와 확실히 차이가 나 동점이 없습니다. 누구인지는 직접 찾으세요).

<details><summary>힌트</summary>

```text
접근방법:
- 각 쌍의 배우 이름을 Counter 로 세고 most_common 으로 1위를 꺼낸다.

세부구현:
1. actor_pairs 의 각 row 에서 row['actor'] 를 모아 Counter 에 넣는다.
2. Counter 에서 가장 많은 원소 하나를 (값, 횟수) 튜플로 돌려주는 메서드로 1위를 꺼내 top_actor 에 담는다.
```

</details>

In [ ]:
# 모든 (배우, 영화) 출연 쌍을 가져옵니다(실행만 하세요).
actor_pairs = run_cypher("MATCH (p:Person)-[:ACTED_IN]->(m:Movie) "
                         "RETURN p.name AS actor, m.title AS movie")
print('출연 관계 수:', len(actor_pairs))

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 그래프에 다시 물어본 1위와 비교합니다.
_row = run_cypher("MATCH (p:Person)-[:ACTED_IN]->(:Movie) "
                  "RETURN p.name AS name, count(*) AS c ORDER BY c DESC LIMIT 1")[0]
assert isinstance(top_actor, tuple) and len(top_actor) == 2, \
    'top_actor 는 (이름, 편수) 튜플 하나여야 합니다(리스트째 담거나 이름만 담지 않았는지 확인하세요)'
assert top_actor == (_row['name'], _row['c']), '다작 배우 1위(이름, 편수)가 다릅니다'
print('✅ 통과!')

## 3. 한 영화에서 여러 배역을 맡은 경우 세기
**배경**: `roles` 는 관계 속성이자 **리스트**라, 한 배우가 한 영화에서 여러 배역을 맡으면 원소가 2개 이상입니다. 그런 (배우, 영화) 조합이 몇 건인지 셉니다.

**요구사항**:
- 제공된 `role_rows`(각 원소에 `roles` 리스트 포함)에서 **`roles` 의 길이가 2 이상**인 출연만 골라 `(배우, 영화)` 튜플의 리스트를 변수 **`multi_pairs`** 에 담으세요. **배우 이름 오름차순**으로 정렬합니다(같으면 영화 제목 오름차순).
- 그 건수를 변수 **`n_multi`** 에 담으세요.

**예시**: `multi_pairs` 의 각 원소는 `('배우 이름', '영화 제목')` 튜플이고, `n_multi` 는 그 목록의 길이입니다(`role_rows` 전체 길이보다 훨씬 작습니다).

<details><summary>힌트</summary>

```text
접근방법:
- roles 리스트 길이가 2 이상인 출연만 골라 (배우, 영화) 튜플 목록으로 만들고, 정렬한 뒤 길이를 센다.

세부구현:
1. role_rows 를 돌며 roles 의 길이가 2 이상인 row 만 고른다.
2. 그 row 의 actor 와 movie 를 튜플로 묶어 모으고, 정렬해 multi_pairs 에 담는다.
3. 그 목록의 길이를 n_multi 에 담는다.
```

</details>

In [ ]:
# 모든 출연의 배우·영화·배역 리스트를 가져옵니다(실행만 하세요).
role_rows = run_cypher("MATCH (p:Person)-[r:ACTED_IN]->(m:Movie) "
                       "RETURN p.name AS actor, m.title AS movie, r.roles AS roles")
print('출연 수:', len(role_rows))

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 제공된 role_rows 로 목록을 다시 만들어 비교합니다(숫자만 적어 넣으면 통과하지 않습니다).
_want = sorted((_r['actor'], _r['movie']) for _r in role_rows if len(_r['roles']) >= 2)
assert all(isinstance(_p, tuple) for _p in multi_pairs), \
    'multi_pairs 의 원소는 (배우, 영화) 튜플이어야 합니다'
assert list(multi_pairs) == _want, \
    '(배우, 영화) 목록이 다릅니다. roles 길이가 2 이상인 출연만 골랐는지, 배우 이름 오름차순으로 정렬했는지 확인하세요'
assert n_multi == len(multi_pairs), 'n_multi 는 multi_pairs 의 길이여야 합니다'
print('✅ 통과!')

## 4. 리뷰어별 리뷰 수 집계
**배경**: `REVIEWED` 는 **사람 → 영화** 방향의 평가 관계입니다. 사람별로 이 관계를 세면 누가 얼마나 활발한 리뷰어인지 한눈에 보입니다. 제공 셀이 (리뷰어, 영화) 쌍을 가져옵니다.

**요구사항**:
- 리뷰어별 리뷰 수를 세어 **`{이름: 리뷰수}` 형태의 dict** 를 변수 **`reviewer_counts`** 에 담으세요(1위만이 아니라 **전원**).
- 값의 타입은 일반 dict 여야 합니다. `Counter` 를 썼다면 `dict(...)` 로 바꿔 담으세요.

**예시**: 리뷰어는 모두 **3명**이고, `reviewer_counts` 는 `{'이름': 리뷰수, ...}` 처럼 사람마다 한 항목씩 담긴 dict 입니다(값은 직접 세세요).

<details><summary>힌트</summary>

```text
접근방법:
- 리뷰어 이름을 세어 dict 로 만든다.

세부구현:
1. review_pairs 의 row['reviewer'] 를 Counter 로 센다.
2. 그 결과를 dict 로 바꿔 reviewer_counts 에 담는다.
```

</details>

In [ ]:
# (리뷰어, 영화) 평가 쌍을 가져옵니다(실행만 하세요).
review_pairs = run_cypher("MATCH (p:Person)-[:REVIEWED]->(m:Movie) "
                          "RETURN p.name AS reviewer, m.title AS movie")
print('리뷰 관계 수:', len(review_pairs))

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 그래프에 다시 물어본 집계와 비교합니다.
_expected = {r['name']: r['c'] for r in run_cypher(
    "MATCH (p:Person)-[:REVIEWED]->(:Movie) RETURN p.name AS name, count(*) AS c")}
assert type(reviewer_counts) is dict, \
    'reviewer_counts 는 일반 dict 여야 합니다(Counter 를 썼다면 dict(...) 로 바꿔 담으세요)'
assert reviewer_counts == _expected, '리뷰어별 리뷰 수 집계가 다릅니다'
print('✅ 통과!')

## 5. 가장 높은 평점을 받은 영화
**배경**: 평가 점수 `rating` 은 `REVIEWED` **관계의 속성**입니다(사람·영화 노드가 아니라 그 평가 자체에 붙음). 이 값으로 최고 평점 리뷰를 찾습니다. 제공 셀이 리뷰를 DataFrame 으로 줍니다.

**요구사항**:
- `reviews_df` 에서 `rating` 이 가장 높은 행의 **영화 제목**을 변수 **`best_movie`** 에, 그 **점수**를 변수 **`best_rating`** 에 담으세요.

**예시**: `best_movie` 는 제목 문자열, `best_rating` 은 정수입니다(최고 평점은 동점이 없습니다).

<details><summary>힌트</summary>

```text
접근방법:
- rating 이 최대인 행을 찾는다(idxmax 로 그 행의 인덱스를 얻어 꺼낸다).

세부구현:
1. rating 열에 idxmax 를 써서 최댓값이 있는 행의 인덱스를 얻고, loc 로 그 행을 꺼낸다.
2. 그 행에서 movie 를 best_movie 에, rating 을 정수로 바꿔 best_rating 에 담는다.
```

</details>

In [ ]:
# 리뷰(리뷰어·영화·점수)를 DataFrame 으로 가져옵니다(실행만 하세요).
reviews_df = pd.DataFrame(run_cypher("MATCH (p:Person)-[r:REVIEWED]->(m:Movie) "
                                     "RETURN p.name AS reviewer, m.title AS movie, r.rating AS rating"))
display(reviews_df)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 그래프에 다시 물어본 최고 평점 리뷰와 비교합니다.
_row = run_cypher("MATCH (:Person)-[r:REVIEWED]->(m:Movie) "
                  "RETURN m.title AS title, r.rating AS rating ORDER BY r.rating DESC LIMIT 1")[0]
assert best_movie == _row['title'], '최고 평점 영화 제목이 다릅니다'
assert best_rating == _row['rating'], '최고 평점 값이 다릅니다'
print('✅ 통과!')

## 6. (서술형) 평점은 왜 관계에 두는가
**배경**: 이 문제는 **자가채점이 없습니다**. 아래 markdown 셀에 서술하고 정답 노트북과 비교하세요.

**요구사항**: 평가 점수 `rating` 을 사람(Person) 노드나 영화(Movie) 노드가 아니라 **`REVIEWED` 관계의 속성**으로 두는 것이 왜 옳은지 설명하세요. (힌트: 한 영화를 여러 사람이 서로 다른 점수로 평가할 수 있습니다.)

**자가 점검**: 답을 쓴 뒤 스스로 확인하세요.
- [ ] "한 영화를 **여러 사람**이 다른 점수로 평가한다"는 상황을 근거로 들었다.
- [ ] 영화 노드에 두면, 사람 노드에 두면 각각 **무엇이 안 되는지** 밝혔다.
- [ ] 답에 **개체(노드)·연결(관계)·딸린 값(속성)** 이라는 낱말이 들어갔다.

*(여기에 자신의 답을 서술하세요)*

---
수고했어요! LV2 에서 제공된 조회 결과를 **pandas·collections 로 가공**해 분포·순위·최고값을 뽑고, 관계 속성의 의미를 서술했습니다. LV3 에서는 **새 도메인의 그래프 모델을 직접 설계**합니다.